In [62]:
import pandas as pd
import numpy as np
import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [63]:
df = pd.read_csv("IMDB Dataset.csv")

df['review'] = df['review'].str.lower()

In [64]:
def remove_html(data):
    if isinstance(data,str):
        import html
        import re 
        data = html.unescape(data)
        data  = re.sub(r'<[^>]+>', ' ',data)
        data  = re.sub(r'\s+', ' ',data)
        return data.strip()
    return data

df['review'] = df['review'].apply(remove_html)

In [65]:
punc = string.punctuation

def remove_pun(data):
    if isinstance(data, str):
        return ''.join(char for char in data if char not in punc)
    return data

df['review'] = df['review'].apply(remove_pun)

In [66]:
df.duplicated().sum()

df.drop_duplicates(inplace=True)

df.duplicated().sum()

0

In [67]:
df['sentiment'].unique()

<StringArray>
['positive', 'negative']
Length: 2, dtype: str

In [68]:
stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    if isinstance(text, str) and text.strip():
        return word_tokenize(text.lower())
    return []

df['tokens'] = df['review'].apply(tokenize_text)

In [69]:
def remove_stop_words(tokens):
    if isinstance(tokens,list):
        return [word for word in tokens if word.lower() not in stop_words]
    return []

df['stop_words_rem'] = df['tokens'].apply(remove_stop_words)

In [70]:
df.columns.to_list()

['review', 'sentiment', 'tokens', 'stop_words_rem']

In [71]:
df['clean_text'] = df['stop_words_rem'].apply(lambda x: ' '.join(x) if x else '')

X = df['clean_text']
y = df['sentiment']

In [72]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [73]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [74]:
tfidf = TfidfVectorizer()

In [75]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [76]:
rn = RandomForestClassifier()
rn.fit(X_train_tfidf,y_train)
y_pred_rn = rn.predict(X_test_tfidf)

In [77]:
print(f"Accuracy Score : {accuracy_score(y_test,y_pred_rn)}")

Accuracy Score : 0.8581081081081081


cv = CountVectorizer(min_df=2,max_features=10000)
X_train_bow = cv.fit_transform(X_train).toarray()  

X_test_bow = cv.transform(X_test).toarray()

gnb = GaussianNB()
gnb.fit(X_train_bow, y_train)

y_pred = gnb.predict(X_test_bow)

print(f"Accuracy Naive Bayes: {accuracy_score(y_test,y_pred)}")

print(f"Confusion Matrix : {confusion_matrix(y_test,y_pred)}")

rn = RandomForestClassifier()
rn.fit(X_train_bow,y_train)
y_pred_rn = rn.predict(X_test_bow)
print(f"Accuracy Random Forests : {accuracy_score(y_test,y_pred_rn)}")